# 🔍 Multimodal RAG System — Complete Walkthrough

This notebook walks through every component of the RAG pipeline:

1. **Document Parsing** — PDF, Image, Web
2. **Semantic Chunking** — Sliding window with overlap
3. **Embedding** — Local TF-IDF or sentence-transformers
4. **Vector Store** — FAISS
5. **Hybrid Retrieval** — BM25 + Dense (RRF fusion)
6. **Reranking** — LLM cross-encoder
7. **Query Rewriting + HyDE** — Better retrieval
8. **Agentic Retrieval** — Multi-step loop
9. **Answer Generation** — Claude
10. **Evaluation** — RAGAS-style metrics

In [ ]:
import os
os.environ['ANTHROPIC_API_KEY'] = 'your_key_here'  # Set your key

import sys
sys.path.insert(0, '..')

## Step 1: Initialize the Pipeline

In [ ]:
from pipeline import RAGPipeline

rag = RAGPipeline()
print('Pipeline ready!')

## Step 2: Ingest Documents

In [ ]:
# Ingest a PDF
# n = rag.ingest('path/to/your/document.pdf')
# print(f'PDF: {n} chunks')

# Ingest a web page
n = rag.ingest('https://en.wikipedia.org/wiki/Retrieval-augmented_generation')
print(f'Web: {n} chunks')

# Check stats
print(rag.stats)

## Step 3: Basic Query

In [ ]:
result = rag.query('What is retrieval-augmented generation?', top_k=3)

print('Answer:')
print(result['answer'])
print('\nSources:')
for s in result['sources']:
    print(f"  [{s['score']:.4f}] {s['source']} (p{s['page']}) — {s['preview'][:100]}...")

## Step 4: Advanced Query with Evaluation

In [ ]:
result = rag.query(
    'What are the main challenges of RAG systems?',
    top_k=5,
    use_agent=True,      # Agentic retrieval loop
    evaluate=True,       # Run RAGAS-style evaluation
    ground_truth='RAG challenges include hallucination, chunking strategy, and retrieval quality.'
)

print('Answer:', result['answer'][:500])
print('\nEvaluation Metrics:')
for k, v in result['eval'].items():
    bar = '█' * int(v * 20) + '░' * (20 - int(v * 20))
    print(f'  {k:<20} [{bar}] {v:.2f}')

## Step 5: Component Deep-Dive — Hybrid Retrieval

In [ ]:
# See exactly what hybrid retrieval returns
query = 'vector database'
docs = rag.retriever.retrieve(query, top_k=5)

print(f'Top results for: "{query}"\n')
for i, doc in enumerate(docs, 1):
    print(f'{i}. RRF={doc.rrf_score:.4f} Dense={doc.dense_score:.3f} BM25={doc.bm25_score:.3f}')
    print(f'   {doc.content[:150]}...')
    print()

## Step 6: Query Rewriting and HyDE

In [ ]:
query = 'how does rag work'

rewritten = rag.llm.rewrite_query(query)
print(f'Original: {query}')
print(f'Rewritten: {rewritten}')

hyde_doc = rag.llm.generate_hypothetical_answer(query)
print(f'\nHyDE document:\n{hyde_doc}')

variants = rag.llm.expand_query(query, n=3)
print(f'\nMulti-query variants:')
for v in variants:
    print(f'  - {v}')

## Step 7: Save and Load

In [ ]:
# Save the vector store
rag.save('./data/my_vector_store')

# Later, load it back
rag2 = RAGPipeline()
rag2.load('./data/my_vector_store')
print(f'Loaded: {rag2.stats}')